# Classificação do Dataset DryBean com KMeans Supervisionado

Este notebook implementa a classificação do dataset DryBean utilizando KMeans supervisionado, com download automático dos dados, avaliação em 30 execuções e salvamento dos resultados (matriz de confusão e gráficos) na pasta `img`.

In [ ]:
# Importar bibliotecas necessárias
import os
import numpy as np
import pandas as pd
from urllib.request import urlretrieve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Carregar e visualizar o dataset DryBean a partir da pasta data
import pandas as pd

csv_path = 'data/Dry_Bean_Dataset.csv'
df = pd.read_csv(csv_path)
df.head()

In [ ]:
# Pré-processamento dos dados
le = LabelEncoder()
df['Class'] = le.fit_transform(df['Class'])
X = df.drop('Class', axis=1)
y = df['Class']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Shape dos dados após preprocessamento: {X_scaled.shape}")

In [ ]:
# Implementação do classificador KMeans supervisionado

def elbow_method(X, max_k=15, random_state=None):
    distortions = []
    K = range(2, max_k+1)
    for k in K:
        kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        kmeans.fit(X)
        distortions.append(kmeans.inertia_)
    deltas = np.diff(distortions)
    elbow = np.argmin(deltas) + 2
    return K[elbow]

class SupervisedKMeansClassifier:
    def __init__(self, max_k=15, random_state=None):
        self.max_k = max_k
        self.random_state = random_state
        self.kmeans = None
        self.n_clusters = None
        self.cluster_labels = None
    def fit(self, X, y):
        self.n_clusters = elbow_method(X, self.max_k, self.random_state)
        self.kmeans = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        clusters = self.kmeans.fit_predict(X)
        self.cluster_labels = np.zeros(self.n_clusters, dtype=int)
        for i in range(self.n_clusters):
            mask = (clusters == i)
            if np.any(mask):
                self.cluster_labels[i] = mode(y[mask], keepdims=True)[0][0]
            else:
                self.cluster_labels[i] = 0
    def predict(self, X):
        clusters = self.kmeans.predict(X)
        return self.cluster_labels[clusters]

In [ ]:
# Treinamento, avaliação e salvamento dos resultados
accs = []
cms = []
for seed in range(1, 31):
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, train_size=0.7, random_state=seed, stratify=y)
    clf = SupervisedKMeansClassifier(random_state=seed)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    accs.append(acc)
    cms.append(cm)
    print(f"Execução {seed:02d}: Acurácia = {acc:.4f}")
accs = np.array(accs)
cms = np.array(cms)

# Salvar matrizes e gráficos
np.save('Trabalho03/img/kmeans_drybean_matrizes_confusao.npy', cms)
np.save('Trabalho03/img/kmeans_drybean_acuracias.npy', accs)

# Gráfico de acurácia
plt.figure(figsize=(8,4))
plt.plot(range(1,31), accs, marker='o')
plt.title('Acurácia em cada execução (KMeans + DryBean)')
plt.xlabel('Execução')
plt.ylabel('Acurácia')
plt.grid()
plt.savefig('Trabalho03/img/kmeans_drybean_acuracia.png')
plt.show()

# Matriz de confusão média
cm_mean = np.round(cms.mean(axis=0)).astype(int)
sns.heatmap(cm_mean, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusão Média (KMeans + DryBean)')
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.savefig('Trabalho03/img/kmeans_drybean_cm_media.png')
plt.show()

print(f"Acurácia média: {accs.mean():.4f}")
print(f"Desvio padrão da acurácia: {accs.std():.4f}")